In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hau100416/now-playing/nowplaying_processed.csv


# 🎵 Music Recommendation Pipeline

Pipeline đầy đủ theo kiến trúc:

```
Item Encoder (track_id + artist + genre + audio_features)
       ↓
Track Embedding [128]
       ↓
GRU User Encoder
       ↓
User Embedding [128]
       ↓
InfoNCE Loss
       ↓
FAISS Retrieval
```

**Các bước trong notebook:**
1. Cài đặt thư viện
2. Tạo dữ liệu mẫu
3. Tiền xử lý dữ liệu
4. Định nghĩa model (Item Encoder + GRU User Encoder + InfoNCE Loss)
5. Training
6. Build FAISS Index
7. Inference — gợi ý bài hát

## 1. Cài đặt thư viện

In [ ]:
!pip install torch faiss-cpu scikit-learn pandas numpy tqdm -q

## 2. Import

In [ ]:
import random
import string
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Tuple
from tqdm import tqdm

import faiss
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 3. Tạo dữ liệu mẫu

> Nếu bạn đã có file dữ liệu thật, bỏ qua cell này và load trực tiếp ở bước 4.

## 4. Tiền xử lý dữ liệu (`DataProcessor`)

- Encode genre → số nguyên, chuẩn hoá audio features (StandardScaler)
- Tạo track feature matrix `[num_tracks, feature_dim]`
- Tạo user sequence: list track_idx theo thứ tự thời gian

In [2]:
df = pd.read_csv('/kaggle/input/datasets/hau100416/now-playing/nowplaying_processed.csv')
df.head()

,user_id,timestamp,track_title,artist_name_1,track_id,popularity,year,genre,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
0,4940d410da0844b24dd96fa22ae461c1db8c7643,'2012-08-30 01:29:17',The Sulphur Man,Doves,7EwrZIIzEEsht1Sb8K38fm,20,2002,electro,0.366,0.743,...,-6.822,1,0.0314,0.01710,0.003350,0.3490,0.2550,100.598,277333,4
1,ff8f057a89184a77ee39211746dac9726540538b,'2012-03-06 00:17:34',Background,Frankyeffe,4hcQI4HMvJmlMbpFOi0Wtv,0,2018,minimal-techno,0.730,0.828,...,-8.791,1,0.0764,0.00267,0.882000,0.0967,0.0377,128.015,410635,4
2,a76d2aeb50403590ada51fc6d588d18d2919fe98,'2012-07-19 12:07:46',Rule,Michael Palascak,0cLRRMNjImZh3LKP9HRPtd,7,2022,comedy,0.417,0.797,...,-15.856,0,0.9350,0.59700,0.000000,0.4520,0.4400,69.935,201108,4
3,ee562ff21f26f8573bed374ce788425226d118f3,'2012-10-09 04:40:13',She Sells Sanctuary,Enuff Z'Nuff,4yPqrjHD1YyftK6fLVCPaE,3,2014,power-pop,0.433,0.844,...,-5.737,1,0.0388,0.00707,0.000504,0.2790,0.4110,134.987,228987,4
4,d0af16b16bbbf351a097b2156bf93020326dccaa,'2012-12-08 06:28:23',Seven Hills,Nym,1DSwu2uyYho34HKYZdoUOn,15,2011,french,0.668,0.770,...,-8.594,0,0.0401,0.03260,0.772000,0.1740,0.3770,92.492,215837,4


In [ ]:
df.shape

In [ ]:
# 1. Đếm số lần xuất hiện của từng user_id
user_counts = df['user_id'].value_counts()

# 2. Lọc lấy danh sách các user_id xuất hiện từ 10 lần trở lên
valid_users = user_counts[user_counts >= 10].index

# 3. Giữ lại các dòng có user_id nằm trong danh sách trên
df_filtered = df[df['user_id'].isin(valid_users)]

In [ ]:
df_filtered.shape

In [ ]:
df_filtered.head()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Lấy ra danh sách tất cả các user_id duy nhất (Unique Users)
unique_users = df_filtered['user_id'].unique()

# 2. Chia danh sách user này thành 2 phần: Train Users (80%) và Test Users (20%)
# Bạn có thể chỉnh tỷ lệ bằng test_size, cố định random_state để kết quả không đổi
train_users, test_users = train_test_split(
    unique_users, 
    test_size=0.2, 
    random_state=42
)

# 3. Lọc dữ liệu từ DataFrame ban đầu dựa trên 2 danh sách user trên
df_train = df_filtered[df_filtered['user_id'].isin(train_users)].reset_index(drop=True)
df_test = df_filtered[df_filtered['user_id'].isin(test_users)].reset_index(drop=True)

# --- KIỂM TRA LẠI KẾT QUẢ ---
print(f"Tổng số User ban đầu: {len(unique_users)}")
print(f"Số lượng User dùng để Train: {len(train_users)} ({len(df_train)} dòng tương tác)")
print(f"Số lượng User để Test sau này: {len(test_users)} ({len(df_test)} dòng tương tác)")

# Đảm bảo chắc chắn không có user nào bị trùng giữa 2 tập
intersect_users = set(df_train['user_id']).intersection(set(df_test['user_id']))
print(f"Số lượng user bị trùng (phải bằng 0): {len(intersect_users)}")

In [ ]:
AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence',
    'tempo', 'duration_ms',
]

class DataProcessor:
    """
    Tiền xử lý dữ liệu nghe nhạc:
      - Chuẩn hoá audio features + encode genre/year/popularity
      - Tạo track feature matrix [num_tracks, feature_dim]
      - Tạo user sequences (list track_idx theo thứ tự thời gian)
    """

    def __init__(self):
        self.genre_encoder  = LabelEncoder()
        self.audio_scaler   = StandardScaler()
        self.track_encoder  = LabelEncoder()   # track_id string → int idx
        self.user_encoder   = LabelEncoder()   # user_id  string → int idx
        self.track_meta: Dict[int, dict] = {}  # idx → metadata
        self.track2idx: Dict[str, int]   = {}  # track_id string → idx
        self.is_fitted = False

    # ── Fit & transform ──────────────────────────────────────────────────────

    def fit_transform(
        self, df: pd.DataFrame
    ) -> Tuple[torch.Tensor, List[List[int]]]:
        """
        Trả về:
          track_features : FloatTensor [num_tracks, feature_dim]
          user_sequences : list of list[int]  (track_idx theo thời gian)
        """
        df = df.copy()

        # Làm sạch timestamp
        df['timestamp'] = (
            df['timestamp'].astype(str).str.strip("'")
        )
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

        # Encode ID → int
        df['track_idx'] = self.track_encoder.fit_transform(df['track_id'])
        df['user_idx']  = self.user_encoder.fit_transform(df['user_id'])

        self.track2idx = dict(zip(
            self.track_encoder.classes_,
            range(len(self.track_encoder.classes_))
        ))

        # Track feature matrix
        track_df = (
            df.drop_duplicates('track_idx')
              .sort_values('track_idx')
              .reset_index(drop=True)
        )

        # Encode genre
        track_df['genre_enc'] = self.genre_encoder.fit_transform(
            track_df['genre'].fillna('unknown')
        )

        # Audio features chuẩn hoá
        avail = [f for f in AUDIO_FEATURES if f in track_df.columns]
        audio_norm = self.audio_scaler.fit_transform(
            track_df[avail].fillna(0).values.astype(np.float32)
        )

        # Popularity, year, genre — chuẩn hoá Z-score
        def _znorm(arr):
            arr = arr.astype(np.float32)
            return ((arr - arr.mean()) / (arr.std() + 1e-8)).reshape(-1, 1)

        pop   = _znorm(track_df['popularity'].fillna(0).values)
        year  = _znorm(track_df['year'].fillna(2000).values)
        genre = _znorm(track_df['genre_enc'].values)

        feat_mat = np.concatenate(
            [audio_norm, pop, year, genre], axis=1
        ).astype(np.float32)

        track_features = torch.tensor(feat_mat)

        # Metadata lookup
        for _, row in track_df.iterrows():
            self.track_meta[int(row['track_idx'])] = {
                'track_id':    row['track_id'],
                'track_title': row.get('track_title', 'Unknown'),
                'artist_name': row.get('artist_name_1',
                               row.get('artist_name', 'Unknown')),
                'genre':       row.get('genre', ''),
                'year':        int(row.get('year', 0)),
                'popularity':  int(row.get('popularity', 0)),
            }

        # User sequences (sắp theo timestamp)
        df = df.sort_values(['user_idx', 'timestamp'])
        user_sequences = [
            grp['track_idx'].tolist()
            for _, grp in df.groupby('user_idx')
        ]

        self.is_fitted = True
        return track_features, user_sequences

    # ── Helpers ──────────────────────────────────────────────────────────────

    def encode_track_ids(self, track_ids: List[str]) -> List[int]:
        return [self.track2idx[t] for t in track_ids if t in self.track2idx]

    def get_meta(self, idx: int) -> dict:
        return self.track_meta.get(idx, {
            'track_id': 'unknown', 'track_title': 'Unknown',
            'artist_name': 'Unknown', 'genre': '',
            'year': 0, 'popularity': 0,
        })

    @property
    def num_tracks(self):  return len(self.track_encoder.classes_)
    @property
    def num_users(self):   return len(self.user_encoder.classes_)
    @property
    def feature_dim(self): return len(AUDIO_FEATURES) + 3


# ── Chạy ────────────────────────────────────────────────────────────────────
processor = DataProcessor()
track_features, user_sequences = processor.fit_transform(df_train)

print(f'Số tracks : {processor.num_tracks}')
print(f'Số users  : {processor.num_users}')
print(f'Feature dim : {processor.feature_dim}')
print(f'track_features shape: {track_features.shape}')
print(f'Độ dài sequence (min/mean/max): '
      f'{min(len(s) for s in user_sequences)} / '
      f'{np.mean([len(s) for s in user_sequences]):.1f} / '
      f'{max(len(s) for s in user_sequences)}')

## 5. Định nghĩa Model

### 5.1 Item Encoder
MLP 3 lớp: `feature_dim → 256 → 256 → 128` với BatchNorm + ReLU + Dropout.

### 5.2 GRU User Encoder
GRU 2 lớp, hidden 256, projection `256 → 128` + LayerNorm.

### 5.3 InfoNCE Loss
Contrastive loss — user embedding vs next-track embedding, negative = các track khác trong batch.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 5.1  Item Encoder
# ════════════════════════════════════════════════════════════════════════════

class ItemEncoder(nn.Module):
    """
    Nhận raw feature vector của track
    → Track Embedding [embedding_dim] (L2-normalized).
    """

    # def __init__(self, feature_dim: int, embedding_dim: int = 128):
    #     super().__init__()
    #     self.net = nn.Sequential(
    #         nn.Linear(feature_dim, 256),
    #         nn.BatchNorm1d(256),
    #         nn.ReLU(),
    #         nn.Dropout(0.2),

    #         nn.Linear(256, 256),
    #         nn.BatchNorm1d(256),
    #         nn.ReLU(),
    #         nn.Dropout(0.2),

    #         nn.Linear(256, embedding_dim),
    #     )

    # def forward(self, x: torch.Tensor) -> torch.Tensor:
    #     # x: [batch, feature_dim] → [batch, embedding_dim]
    #     return F.normalize(self.net(x), dim=-1)

    def __init__(self, feature_dim: int, embedding_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.LayerNorm(256),  # <--- ĐỔI TẠI ĐÂY: Xử lý mượt cả chuỗi 3D lẫn 2D
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 256),
            nn.LayerNorm(256),  # <--- ĐỔI TẠI ĐÂY
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, embedding_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), dim=-1)
# ════════════════════════════════════════════════════════════════════════════
# 5.2  GRU User Encoder
# ════════════════════════════════════════════════════════════════════════════

class GRUUserEncoder(nn.Module):
    """
    Nhận sequence track embeddings theo lịch sử nghe
    → User Embedding [embedding_dim] (L2-normalized).
    """

    def __init__(self, embedding_dim: int = 128,
                 hidden_dim: int = 256, num_layers: int = 2,
                 dropout: float = 0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, embedding_dim),
            nn.LayerNorm(embedding_dim),
        )

    def forward(self, seq: torch.Tensor,
                lengths: torch.Tensor) -> torch.Tensor:
        """
        seq    : [batch, max_len, embedding_dim]
        lengths: [batch]  — độ dài thật của mỗi sequence
        → [batch, embedding_dim]
        """
        packed = nn.utils.rnn.pack_padded_sequence(
            seq, lengths.cpu(), batch_first=True,
            enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        last_hidden = hidden[-1]           # [batch, hidden_dim]
        return F.normalize(self.proj(last_hidden), dim=-1)


# ════════════════════════════════════════════════════════════════════════════
# 5.3  InfoNCE Loss
# ════════════════════════════════════════════════════════════════════════════

class InfoNCELoss(nn.Module):
    """
    L = -log [ exp(sim(u_i, t+_i)/τ) / Σ_j exp(sim(u_i, t_j)/τ) ]

    Positive pair : (user_emb[i], pos_track_emb[i])
    Negatives     : tất cả track khác trong cùng batch
    """

    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, user_emb: torch.Tensor,
                pos_track_emb: torch.Tensor) -> torch.Tensor:
        # user_emb, pos_track_emb: [batch, dim], đã L2-normalize
        batch_size = user_emb.size(0)
        sim    = torch.matmul(user_emb, pos_track_emb.T) / self.temperature
        labels = torch.arange(batch_size, device=user_emb.device)
        return F.cross_entropy(sim, labels)


# ════════════════════════════════════════════════════════════════════════════
# 5.4  Full Model
# ════════════════════════════════════════════════════════════════════════════

class MusicRecommender(nn.Module):
    """Ghép ItemEncoder + GRUUserEncoder + InfoNCELoss thành một module."""

    def __init__(self, feature_dim: int, embedding_dim: int = 128,
                 temperature: float = 0.07):
        super().__init__()
        self.item_encoder = ItemEncoder(feature_dim, embedding_dim)
        self.user_encoder = GRUUserEncoder(embedding_dim)
        self.loss_fn      = InfoNCELoss(temperature)
        self.embedding_dim = embedding_dim

    # ── Inference helpers ────────────────────────────────────────────────────

    @torch.no_grad()
    def encode_catalog(self, track_features: torch.Tensor,
                       batch_size: int = 512) -> torch.Tensor:
        """Encode toàn bộ track catalog → [num_tracks, embedding_dim]."""
        self.item_encoder.eval()
        parts = []
        for i in range(0, len(track_features), batch_size):
            parts.append(self.item_encoder(track_features[i:i+batch_size]))
        return torch.cat(parts, dim=0)

    @torch.no_grad()
    def encode_user_from_history(
        self, history_idxs: List[int],
        track_features: torch.Tensor
    ) -> torch.Tensor:
        """Encode 1 user từ list track_idx → [1, embedding_dim]."""
        self.eval()
        history_idxs = history_idxs[-20:]
        hist_feats = track_features[history_idxs]          # [seq, feat_dim]
        hist_embs  = self.item_encoder(hist_feats)          # [seq, emb_dim]
        seq    = hist_embs.unsqueeze(0)                     # [1, seq, emb_dim]
        length = torch.tensor([hist_embs.size(0)])
        return self.user_encoder(seq, length)               # [1, emb_dim]

    # ── Training forward ─────────────────────────────────────────────────────

    # def forward(
    #     self,
    #     seq_embeddings:   torch.Tensor,   # [batch, max_len, emb_dim]
    #     seq_lengths:      torch.Tensor,   # [batch]
    #     pos_track_feats:  torch.Tensor,   # [batch, feat_dim]
    # ) -> torch.Tensor:
    #     user_emb      = self.user_encoder(seq_embeddings, seq_lengths)
    #     pos_track_emb = self.item_encoder(pos_track_feats)
    #     return self.loss_fn(user_emb, pos_track_emb)
    def forward(
        self,
        seq_feats:       torch.Tensor,   # Nhận raw features [batch, max_len, 13] từ DataLoader mới
        seq_lengths:     torch.Tensor,   # [batch]
        pos_track_feats: torch.Tensor,   # [batch, 13]
    ) -> torch.Tensor:
        
        # 1. Đi qua item_encoder để biến đổi chuỗi feature (13) -> chuỗi embedding (128)
        seq_embeddings = self.item_encoder(seq_feats) # Output lúc này sẽ chuẩn kích thước [batch, max_len, 128]
        
        # 2. Đưa chuỗi embedding chuẩn vào GRU
        user_emb      = self.user_encoder(seq_embeddings, seq_lengths)
        
        # 3. Tạo Target Track Embedding
        pos_track_emb = self.item_encoder(pos_track_feats)
        
        # 4. Tính toán Loss
        return self.loss_fn(user_emb, pos_track_emb)


# ── Khởi tạo ────────────────────────────────────────────────────────────────
EMBEDDING_DIM = 128
TEMPERATURE   = 0.07

model = MusicRecommender(
    feature_dim=processor.feature_dim,
    embedding_dim=EMBEDDING_DIM,
    temperature=TEMPERATURE,
).to(DEVICE)


total_params = sum(p.numel() for p in model.parameters())
print(f'Tổng tham số: {total_params:,}')
print(model)

## 6. Training

### Dataset
Sliding-window trên user sequence: `[t0…t_{k−1}]` → input, `t_k` → positive track.

### Tối ưu
AdamW + Cosine Annealing LR + gradient clipping.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Dataset — Sliding window
# ════════════════════════════════════════════════════════════════════════════

class SessionDataset(Dataset):
    """
    Mỗi sample: (history_idx_list, positive_track_idx)
    Tạo bằng cách trượt cửa sổ trên user sequence.
    """

    def __init__(
        self,
        user_sequences: List[List[int]],
        min_seq_len: int = 2,
        max_seq_len: int = 50,
    ):
        self.samples: List[Tuple[List[int], int]] = []
        for seq in user_sequences:
            if len(seq) < min_seq_len:
                continue
            for end in range(min_seq_len, len(seq) + 1):
                history  = seq[max(0, end - max_seq_len): end - 1]
                pos_track = seq[end - 1]
                if history:
                    self.samples.append((history, pos_track))

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ════════════════════════════════════════════════════════════════════════════
# Collate — padding + encode tracks → embeddings
# ════════════════════════════════════════════════════════════════════════════

# def make_collate_fn(track_features: torch.Tensor, item_encoder: ItemEncoder):
#     """Closure giữ track_features và item_encoder."""

    # def collate_fn(batch):
    #     histories, pos_tracks = zip(*batch)
    #     lengths = torch.tensor([len(h) for h in histories])
    #     max_len = int(lengths.max())
    #     emb_dim = item_encoder.net[-1].out_features

    #     # Encode tất cả track cần thiết (no grad)
    #     all_idxs = list({idx for h in histories for idx in h})
    #     item_encoder.eval()
    #     with torch.no_grad():
    #         chunk_embs = item_encoder(track_features[all_idxs])
    #     idx2emb = {idx: emb for idx, emb in zip(all_idxs, chunk_embs)}

    #     # Pad sequences
    #     seq_emb = torch.zeros(len(histories), max_len, emb_dim)
    #     for i, hist in enumerate(histories):
    #         for j, tidx in enumerate(hist):
    #             seq_emb[i, j] = idx2emb[tidx]

    #     pos_feats = track_features[[t for t in pos_tracks]]
    #     return seq_emb, lengths, pos_feats

    # return collate_fn
    # def collate_fn(batch):
    #     histories, pos_tracks = zip(*batch)
    #     lengths = torch.tensor([len(h) for h in histories])
    #     max_len = int(lengths.max())
    #     emb_dim = item_encoder.net[-1].out_features

    #     # Encode tất cả track cần thiết (no grad)
    #     all_idxs = list({idx for h in histories for idx in h})
    #     item_encoder.eval()
        
    #     # FIX: Lấy device hiện tại của item_encoder (cuda:0 hoặc cpu)
    #     device = next(item_encoder.parameters()).device
        
    #     with torch.no_grad():
    #         # FIX: Chuyển dữ liệu đầu vào lên cùng device với encoder, 
    #         # sau đó đưa kết quả .cpu() về lại để mapping vào seq_emb
    #         chunk_embs = item_encoder(track_features[all_idxs].to(device)).cpu()
            
    #     idx2emb = {idx: emb for idx, emb in zip(all_idxs, chunk_embs)}

    #     # Pad sequences
    #     seq_emb = torch.zeros(len(histories), max_len, emb_dim)
    #     for i, hist in enumerate(histories):
    #         for j, tidx in enumerate(hist):
    #             seq_emb[i, j] = idx2emb[tidx]

    #     pos_feats = track_features[[t for t in pos_tracks]]
    #     return seq_emb, lengths, pos_feats

    # return collate_fn

def make_collate_fn(track_features: torch.Tensor):
    """
    Closure chỉ giữ track_features gốc. 
    Không cần truyền item_encoder vào đây nữa!
    """
    def collate_fn(batch):
        histories, pos_tracks = zip(*batch)
        lengths = torch.tensor([len(h) for h in histories])
        max_len = int(lengths.max())
        feat_dim = track_features.size(1) # Kích thước đặc trưng gốc (13)

        # 1. Khởi tạo tensor chứa RAW features [batch, max_len, feat_dim] điền sẵn số 0 (padding)
        seq_feats = torch.zeros(len(histories), max_len, feat_dim)
        
        # 2. Đổ dữ liệu đặc trưng vào (Vận hành theo hàng, bỏ được 1 vòng lặp for phía trong -> Cực nhanh)
        for i, hist in enumerate(histories):
            seq_feats[i, :len(hist)] = track_features[hist]

        # 3. Lấy đặc trưng của track mục tiêu
        pos_feats = track_features[list(pos_tracks)]
        
        # Trả về raw features dạng chuỗi, độ dài thật, và raw features của bài hát đích
        return seq_feats, lengths, pos_feats

    return collate_fn
# ════════════════════════════════════════════════════════════════════════════
# Training loop
# ════════════════════════════════════════════════════════════════════════════

# def train(
#     model: MusicRecommender,
#     track_features: torch.Tensor,
#     user_sequences: List[List[int]],
#     epochs:     int   = 10,
#     batch_size: int   = 64,
#     lr:         float = 1e-3,
# ) -> List[float]:

#     tf = track_features.to(DEVICE)
#     model.to(DEVICE)

#     dataset = SessionDataset(user_sequences)
#     print(f'Training samples: {len(dataset):,}')

#     loader = DataLoader(
#         dataset, batch_size=batch_size, shuffle=True,
#         collate_fn=make_collate_fn(tf.cpu(), model.item_encoder),
#         num_workers=0,
#     )

#     optimizer = torch.optim.AdamW(
#         model.parameters(), lr=lr, weight_decay=1e-4
#     )
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#         optimizer, T_max=epochs, eta_min=lr * 0.1
#     )

#     loss_history = []

#     for epoch in range(1, epochs + 1):
#         model.train()
#         total_loss, n_batches = 0.0, 0

#         for seq_emb, seq_len, pos_feat in loader:
#             seq_emb  = seq_emb.to(DEVICE)
#             seq_len  = seq_len.to(DEVICE)
#             pos_feat = pos_feat.to(DEVICE)

#             optimizer.zero_grad()
#             loss = model(seq_emb, seq_len, pos_feat)
#             loss.backward()
#             nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#             optimizer.step()

#             total_loss += loss.item()
#             n_batches  += 1

#         scheduler.step()
#         avg = total_loss / max(n_batches, 1)
#         loss_history.append(avg)
#         print(f'  Epoch {epoch:>3}/{epochs}  loss = {avg:.4f}')

#     return loss_history

def train(
    model: MusicRecommender,
    track_features: torch.Tensor,
    user_sequences: List[List[int]],
    epochs:     int   = 10,
    batch_size: int   = 256,  # Tăng lên 256 để tận dụng sức mạnh của T4
    lr:         float = 1e-3,
) -> List[float]:

    # Đảm bảo track_features gốc nằm trên CPU tại DataLoader để tránh xung đột luồng
    track_features_cpu = track_features.cpu()
    model.to(DEVICE)

    # Khởi tạo dataset cắt chuỗi tối đa 50 (max_seq_len=50 như bạn đã cấu hình mặc định)
    dataset = SessionDataset(user_sequences, max_seq_len=21)
    print(f'Training samples: {len(dataset):,}')

    # Cấu hình DataLoader tối ưu cho Kaggle
    loader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True,
        collate_fn=make_collate_fn(track_features_cpu),
        num_workers=2,         # Kaggle cung cấp 2 CPU, hãy tận dụng để load data song song
        pin_memory=True        # Tăng tốc độ truyền dữ liệu từ RAM CPU lên VRAM GPU
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr * 0.1
    )

    loss_history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, n_batches = 0.0, 0

        # Đổi tên từ seq_emb thành seq_feats cho đúng bản chất dữ liệu đầu vào
        for seq_feats, seq_len, pos_feat in tqdm(loader):
            # Đẩy toàn bộ raw data lên GPU cùng lúc
            seq_feats = seq_feats.to(DEVICE)
            seq_len   = seq_len.to(DEVICE)
            pos_feat  = pos_feat.to(DEVICE)

            optimizer.zero_grad()
            
            # Lúc này Model nhận vào Raw Features, tự đi qua ItemEncoder bên trong forward()
            # Toàn bộ lịch sử nghe sẽ được tính Gradient hoàn chỉnh!
            loss = model(seq_feats, seq_len, pos_feat)
            
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

        scheduler.step()
        avg = total_loss / max(n_batches, 1)
        loss_history.append(avg)
        print(f'  Epoch {epoch:>3}/{epochs}  loss = {avg:.4f}')

    return loss_history

# ── Chạy training ────────────────────────────────────────────────────────────
EPOCHS     = 10
BATCH_SIZE = 256
LR         = 1e-3

print('Bắt đầu training...')
loss_history = train(
    model, track_features, user_sequences,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
)
print(f'\nFinal loss: {loss_history[-1]:.4f}')

In [ ]:
# Vẽ loss curve
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3))
plt.plot(range(1, len(loss_history)+1), loss_history,
         marker='o', linewidth=2, color='steelblue')
plt.title('InfoNCE Loss theo epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Build FAISS Index

Encode toàn bộ track catalog → FAISS `IndexFlatIP` (inner product = cosine similarity sau L2 normalize).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FAISS Retriever
# ════════════════════════════════════════════════════════════════════════════

class FAISSRetriever:
    """
    Wrapper FAISS IndexFlatIP:
      build(embeddings) — nạp toàn bộ track embeddings
      search(query, k)  — top-k nearest neighbors
    """

    def __init__(self, embedding_dim: int = 128):
        self.embedding_dim = embedding_dim
        self.index: faiss.Index = None

    def build(self, track_embeddings: torch.Tensor) -> None:
        """track_embeddings: [num_tracks, embedding_dim], đã L2-normalize."""
        vecs = track_embeddings.cpu().numpy().astype(np.float32)
        self.index = faiss.IndexFlatIP(self.embedding_dim)
        self.index.add(vecs)
        print(f'FAISS index built: {self.index.ntotal:,} tracks')

    def search(
        self, query: torch.Tensor, k: int = 10,
        exclude_ids: List[int] = None,
    ) -> List[Tuple[int, float]]:
        """
        query       : [embedding_dim] hoặc [1, embedding_dim]
        exclude_ids : track_idx cần loại bỏ (đã nghe)
        → list of (track_idx, score)
        """
        q = query.cpu().numpy().astype(np.float32)
        if q.ndim == 1:
            q = q.reshape(1, -1)

        k_fetch = k + len(exclude_ids or []) + 10
        k_fetch = min(k_fetch, self.index.ntotal)

        scores, indices = self.index.search(q, k_fetch)
        exclude_set = set(exclude_ids or [])

        results = []
        for idx, score in zip(indices[0], scores[0]):
            if idx < 0 or idx in exclude_set:
                continue
            results.append((int(idx), float(score)))
            if len(results) >= k:
                break
        return results

    @property
    def size(self): return self.index.ntotal if self.index else 0


# ── Build ────────────────────────────────────────────────────────────────────
retriever = FAISSRetriever(embedding_dim=EMBEDDING_DIM)

print('Đang encode catalog...')
model.eval()
track_embeddings = model.encode_catalog(track_features.to(DEVICE))
print(f'Track embeddings shape: {track_embeddings.shape}')

retriever.build(track_embeddings)

# Tạo user sequences map: user_id_str → list track_idx
user_seq_map: Dict[str, List[int]] = {
    uid: seq
    for uid, seq in zip(processor.user_encoder.classes_, user_sequences)
}

## 8. Inference — Gợi ý bài hát

In [ ]:
def recommend(
    history_idxs: List[int],
    top_k: int = 10,
    label: str = '',
) -> pd.DataFrame:
    """
    Nhận list track_idx lịch sử → gợi ý top_k bài tiếp theo.
    Trả về DataFrame để hiển thị dễ đọc.
    """
    user_emb = model.encode_user_from_history(
        history_idxs, track_features.to(DEVICE)
    )
    results = retriever.search(user_emb.squeeze(0), k=top_k,
                               exclude_ids=history_idxs)

    rows = []
    for rank, (idx, score) in enumerate(results, 1):
        meta = processor.get_meta(idx)
        rows.append({
            'rank':         rank,
            'score':        round(score, 4),
            'track_title':  meta['track_title'],
            'artist':       meta['artist_name'],
            'genre':        meta['genre'],
            'year':         meta['year'],
            'popularity':   meta['popularity'],
            'track_id':     meta['track_id'],
        })

    df_result = pd.DataFrame(rows)
    if label:
        print(f'\n── {label} ──')
    return df_result

In [ ]:
# ── Ví dụ 1: Gợi ý cho user trong training data ──────────────────────────────

sample_user_id = processor.user_encoder.classes_[0]
history_idxs   = user_seq_map[sample_user_id]

print(f'User ID  : {sample_user_id[:20]}...')
print(f'Lịch sử  : {len(history_idxs)} bài đã nghe')

# Hiển thị 5 bài gần nhất đã nghe
last5 = history_idxs[-5:]
print('\n5 bài nghe gần nhất:')
for idx in last5:
    m = processor.get_meta(idx)
    print(f'  [{idx}] {m["track_title"]}  –  {m["artist_name"]}  ({m["genre"]})')

rec_df = recommend(history_idxs, top_k=10,
                   label=f'Gợi ý cho user {sample_user_id[:16]}…')
rec_df

In [ ]:
# ── Ví dụ 2: Cold-start — gợi ý từ list track_id bất kỳ ─────────────────────

# Lấy 3 track_id ngẫu nhiên từ catalog
seed_track_ids = list(processor.track2idx.keys())[:3]
seed_idxs      = processor.encode_track_ids(seed_track_ids)

print('Seed tracks:')
for sid in seed_idxs:
    m = processor.get_meta(sid)
    print(f'  {m["track_title"]}  –  {m["artist_name"]}  ({m["genre"]})')

cold_df = recommend(seed_idxs, top_k=10, label='Cold-start recommendation')
cold_df

In [ ]:
# ── Ví dụ 3: Trực quan hoá phân phối genre được gợi ý ────────────────────────

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (df, title) in zip(axes, [
    (rec_df,  'Gợi ý cho existing user'),
    (cold_df, 'Cold-start recommendation'),
]):
    counts = df['genre'].value_counts()
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Genre')
    ax.set_ylabel('Số bài')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 9. Lưu & Load model

In [ ]:
import pickle, os, json as _json

SAVE_DIR = '/kaggle/working/saved_model'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 1. Lưu config model (để load lại không cần hard-code) ───────────────────
model_config = {
    'feature_dim'   : processor.feature_dim,
    'embedding_dim' : EMBEDDING_DIM,
    'temperature'   : TEMPERATURE,
    'audio_features': AUDIO_FEATURES,
}
with open(f'{SAVE_DIR}/model_config.json', 'w') as _f:
    _json.dump(model_config, _f, indent=2)

# ── 2. Lưu model weights ────────────────────────────────────────────────────
torch.save(model.state_dict(), f'{SAVE_DIR}/model.pt')

# ── 3. Lưu FAISS index ──────────────────────────────────────────────────────
faiss.write_index(retriever.index, f'{SAVE_DIR}/faiss.index')

# ── 4. Lưu track_features tensor (cần cho inference) ───────────────────────
torch.save(track_features.cpu(), f'{SAVE_DIR}/track_features.pt')

# ── 5. Lưu processor (encoders + scaler + metadata) ────────────────────────
with open(f'{SAVE_DIR}/processor.pkl', 'wb') as _f:
    pickle.dump(processor, _f)

print('✅ Đã lưu đầy đủ artifacts vào', SAVE_DIR)
print('   model_config.json  — hyperparameters')
print('   model.pt           — model weights')
print('   faiss.index        — FAISS index')
print('   track_features.pt  — track feature matrix')
print('   processor.pkl      — encoders + scaler + metadata')

# ── Kiểm tra load lại ngay trong notebook ───────────────────────────────────
import json as _json2

with open(f'{SAVE_DIR}/model_config.json') as _f:
    _cfg = _json2.load(_f)

_model = MusicRecommender(
    feature_dim=_cfg['feature_dim'],
    embedding_dim=_cfg['embedding_dim'],
    temperature=_cfg['temperature'],
).to(DEVICE)
_model.load_state_dict(torch.load(f'{SAVE_DIR}/model.pt', map_location=DEVICE))
_model.eval()

_retriever = FAISSRetriever(_cfg['embedding_dim'])
_retriever.index = faiss.read_index(f'{SAVE_DIR}/faiss.index')

with open(f'{SAVE_DIR}/processor.pkl', 'rb') as _f:
    _processor = pickle.load(_f)

_track_features = torch.load(f'{SAVE_DIR}/track_features.pt', map_location='cpu')

print(f'\n✅ Load thành công!')
print(f'   Tracks  : {_processor.num_tracks:,}')
print(f'   FAISS   : {_retriever.size:,} vectors')
print(f'   Feature : {_track_features.shape}')
